In [ ]:
import os
import json
import re
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel
from google import genai
from google.genai import types

load_dotenv()  # picks up GEMINI_API_KEY from a local .env file, if present


# STRUCTURED OUTPUT SCHEMA
# The model never free-forms the final answer -- once the agent loop has
# collected real tool results, this code assembles them into
# a validated Pydantic object, so the output is guaranteed to be parseable.

class FlightInfo(BaseModel):
    destination: str
    price: float

class HotelInfo(BaseModel):
    hotel_name: str
    price_per_night: float
    total_nights_cost: float

class TripItinerary(BaseModel):
    status: str = "success"
    destination: str
    total_budget: float
    total_actual_cost: float
    under_budget: bool
    flight: FlightInfo
    hotel: HotelInfo


# TOOLS
# which can hallucinate or be steered by injected text, so each tool
# validates and sanitizes its own arguments before touching any data.

def search_flights(destination: str) -> Dict[str, Any]:
    # strip anything that isn't a letter or space -- catches path traversal /
    # injection attempts like '../../etc/passwd' or 'porto; DROP TABLE' before
    # they reach the lookup
    clean = re.sub(r'[^a-zA-Z\s]', '', str(destination)).strip().lower()

    mock_flights = {
        "porto": {"destination": "Porto (OPO)", "price": 180.0},
        "baku":  {"destination": "Baku (GYD)",  "price": 250.0},
    }

    if clean in mock_flights:
        return mock_flights[clean]
    return {"error": f"No flights found for '{destination}'"}


def search_hotels(destination: str, nights: int = 3) -> Dict[str, Any]:
    clean = re.sub(r'[^a-zA-Z\s]', '', str(destination)).strip().lower()

    try:
        nights = int(nights)
    except (ValueError, TypeError):
        return {"error": "nights must be an integer"}

    # reject floats-disguised-as-ints, negatives, and absurdly large values
    # that would blow up the cost calculation
    if nights <= 0 or nights > 30:
        return {"error": "nights must be an integer between 1 and 30"}

    mock_hotels = {
        "porto": {"hotel_name": "Ribeira Douro Hotel", "price_per_night": 90.0},
        "baku":  {"hotel_name": "Caspian Breeze Hotel", "price_per_night": 80.0},
    }

    if clean in mock_hotels:
        hotel = mock_hotels[clean]
        total = hotel["price_per_night"] * nights
        return {
            "hotel_name":        hotel["hotel_name"],
            "price_per_night":   hotel["price_per_night"],
            "total_nights_cost": total,
        }
    return {"error": f"No hotels found for '{destination}'"}


def calculate_total(flight_cost: float, hotel_cost: float) -> Dict[str, Any]:
    # cast to float explicitly so string injections like '1e999' or 'NaN'
    # get caught right here rather than silently corrupting the final output
    try:
        f = float(flight_cost)
        h = float(hotel_cost)
    except (ValueError, TypeError):
        return {"error": "both costs must be valid numbers"}

    if f != f or h != h:  # NaN check
        return {"error": "costs cannot be NaN"}
    if f < 0 or h < 0:
        return {"error": "costs can't be negative"}

    return {"total_cost": round(f + h, 2)}


TOOLS = {
    "search_flights":  search_flights,
    "search_hotels":   search_hotels,
    "calculate_total": calculate_total,
}

TOOL_DECLARATIONS = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_flights",
        description="Look up the base flight price to a destination city.",
        parameters=types.Schema(
            type="OBJECT",
            properties={"destination": types.Schema(type="STRING")},
            required=["destination"],
        ),
    ),
    types.FunctionDeclaration(
        name="search_hotels",
        description="Look up the total hotel cost for a destination and number of nights.",
        parameters=types.Schema(
            type="OBJECT",
            properties={
                "destination": types.Schema(type="STRING"),
                "nights": types.Schema(type="INTEGER"),
            },
            required=["destination", "nights"],
        ),
    ),
    types.FunctionDeclaration(
        name="calculate_total",
        description="Add a flight cost and a hotel cost together to get the trip total.",
        parameters=types.Schema(
            type="OBJECT",
            properties={
                "flight_cost": types.Schema(type="NUMBER"),
                "hotel_cost": types.Schema(type="NUMBER"),
            },
            required=["flight_cost", "hotel_cost"],
        ),
    ),
])


# AGENT
# The model decides, turn by turn, which tool to call and with what
# arguments based on what it has seen so far -- there is no fixed call


class MultiToolAgent:
    def __init__(self, model: str = "gemini-2.5-flash", step_limit: int = 5):
        # the task needs exactly 3 tool calls to complete, so 5 gives one
        # retry buffer for a bad/failed call without letting the model loop
        # forever and burn API quota
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError(
                "GEMINI_API_KEY is not set. Put it in a local .env file "
                "(GEMINI_API_KEY=...) next to this notebook, or export it "
                "in your shell before launching Jupyter."
            )
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.step_limit = step_limit

    def run(self, goal: str) -> str:
        chat = self.client.chats.create(
            model=self.model,
            config=types.GenerateContentConfig(
                tools=[TOOL_DECLARATIONS],
                system_instruction=(
                    "You are a trip-planning agent. You have three tools: "
                    "search_flights, search_hotels, and calculate_total. "
                    "Call whichever tools you need, in whatever order makes sense, "
                    "to gather the flight cost and hotel cost for the requested trip "
                    "and compute the total with calculate_total. "
                    "Once calculate_total has returned a total, reply with plain text "
                    "summarizing that you are done -- do not call any more tools."
                ),
            ),
        )

        print(f"Starting Agent with Goal: '{goal}'")
        print(f"Safety Check: Active. Max Step Limit: {self.step_limit}")
        print("-" * 50)

        collected: Dict[str, Any] = {"flight_data": None, "hotel_data": None, "total_cost": None}
        message: Any = goal

        for step in range(1, self.step_limit + 1):
            print(f"\n[Step {step}/{self.step_limit}] Calling model...")
            response = chat.send_message(message)

            calls = response.function_calls or []
            if not calls:
                print(f"Model responded with no further tool calls: {response.text!r}")
                break

            call = calls[0]
            tool_name = call.name
            tool_args = dict(call.args)
            print(f"Model chose tool: '{tool_name}' with args: {tool_args}")

            if tool_name not in TOOLS:
                print(f"Model requested unknown tool '{tool_name}' -- refusing to call it.")
                return json.dumps({"status": "failed", "reason": f"unknown tool '{tool_name}'"})

            try:
                result = TOOLS[tool_name](**tool_args)
            except TypeError as exc:
                print(f"Model supplied malformed args for '{tool_name}': {exc}")
                return json.dumps({"status": "failed", "reason": f"malformed arguments for '{tool_name}'"})

            if "error" in result:
                print(f"Tool reported error: {result['error']}")
                return json.dumps({"status": "failed", "reason": result["error"]})

            print(f"Observation (Tool Output): {result}")

            if tool_name == "search_flights":
                collected["flight_data"] = result
            elif tool_name == "search_hotels":
                collected["hotel_data"] = result
            elif tool_name == "calculate_total":
                collected["total_cost"] = result["total_cost"]

            message = types.Part.from_function_response(name=tool_name, response={"result": result})
        else:
            print("Step limit reached before the agent finished.")
            return json.dumps({"status": "failed", "reason": "step limit exceeded"})

        if not (collected["flight_data"] and collected["hotel_data"] and collected["total_cost"] is not None):
            return json.dumps({"status": "failed", "reason": "agent stopped before gathering all required data"})

        budget = 600.0
        itinerary = TripItinerary(
            destination=collected["flight_data"]["destination"].split(" (")[0],
            total_budget=budget,
            total_actual_cost=collected["total_cost"],
            under_budget=collected["total_cost"] <= budget,
            flight=FlightInfo(**collected["flight_data"]),
            hotel=HotelInfo(**collected["hotel_data"]),
        )
        return itinerary.model_dump_json(indent=2)


# RUN

agent = MultiToolAgent(step_limit=5)
goal = "Plan a 3-day trip to Porto under EUR600 and give me the total."
output = agent.run(goal)

print("\n================ FINAL STRUCTURED OUTPUT ================")
print(output)


Starting Agent with Goal: 'Plan a 3-day trip to Porto under EUR600 and give me the total.'
Safety Check: Active. Max Step Limit: 5
--------------------------------------------------

[Step 1/5] Calling model...
Model chose tool: 'search_flights' with args: {'destination': 'Porto'}
Observation (Tool Output): {'destination': 'Porto (OPO)', 'price': 180.0}

[Step 2/5] Calling model...
Model chose tool: 'search_hotels' with args: {'destination': 'Porto', 'nights': 3}
Observation (Tool Output): {'hotel_name': 'Ribeira Douro Hotel', 'price_per_night': 90.0, 'total_nights_cost': 270.0}

[Step 3/5] Calling model...
Model chose tool: 'calculate_total' with args: {'flight_cost': 180, 'hotel_cost': 270}
Observation (Tool Output): {'total_cost': 450.0}

[Step 4/5] Calling model...
Model responded with no further tool calls: 'The total cost for your 3-day trip to Porto is EUR450.\n'

================ FINAL STRUCTURED OUTPUT ================
{
  "status": "success",
  "destination": "Porto",
  "tota

In [3]:
# SECOND RUN: a different destination/order, to show the model isn't
# following a fixed script -- it genuinely re-decides args and sequencing
# each time based on the goal text, not on hardcoded logic in this notebook.

goal3 = "I'm going to Baku for 5 nights, what's my hotel cost and flight cost, and what's the total?"
output3 = agent.run(goal3)

print("================ FINAL STRUCTURED OUTPUT (run 3) ================")
print(output3)


Starting Agent with Goal: 'I'm going to Baku for 5 nights, what's my hotel cost and flight cost, and what's the total?'
Safety Check: Active. Max Step Limit: 5
--------------------------------------------------

[Step 1/5] Calling model...
Model chose tool: 'search_flights' with args: {'destination': 'Baku'}
Observation (Tool Output): {'destination': 'Baku (GYD)', 'price': 250.0}

[Step 2/5] Calling model...
Model chose tool: 'search_hotels' with args: {'nights': 5, 'destination': 'Baku'}
Observation (Tool Output): {'hotel_name': 'Caspian Breeze Hotel', 'price_per_night': 80.0, 'total_nights_cost': 400.0}

[Step 3/5] Calling model...
Model chose tool: 'calculate_total' with args: {'flight_cost': 250, 'hotel_cost': 400}
Observation (Tool Output): {'total_cost': 650.0}

[Step 4/5] Calling model...
Model responded with no further tool calls: 'I am done. Your flight cost to Baku is 250, your hotel cost for 5 nights in Baku is 400, and your total trip cost is 650.\n'
================ FINAL 

In [3]:
# THIRD RUN: failure path actually executed (not just described).
# Porto/Baku are the only destinations in the mock data, so asking for a
# city the tools don't know about makes search_flights genuinely return
# {"error": ...}, which the agent loop turns into a structured failure
# instead of crashing or hallucinating a price.

goal3 = "Plan a 3-day trip to Atlantis under EUR600 and give me the total."
output3 = agent.run(goal3)

print("================ FINAL STRUCTURED OUTPUT (run 3, failure path) ================")
print(output3)


Starting Agent with Goal: 'Plan a 3-day trip to Atlantis under EUR600 and give me the total.'
Safety Check: Active. Max Step Limit: 5
--------------------------------------------------

[Step 1/5] Calling model...
Model chose tool: 'search_flights' with args: {'destination': 'Atlantis'}
Tool reported error: No flights found for 'Atlantis'
================ FINAL STRUCTURED OUTPUT (run 3, failure path) ================
{"status": "failed", "reason": "No flights found for 'Atlantis'"}


In [2]:
# FOURTH RUN: deterministically force the step-limit path to fire.
# Rather than hoping a real failure happens to land exactly on the cap,
# this re-creates the agent with an unrealistically low step_limit=1 --
# the task needs 3 tool calls, so it cannot finish in 1, which proves the
# guard rail itself actually runs (not just that it's described).

small_agent = MultiToolAgent(step_limit=1)
goal4 = "Plan a 3-day trip to Porto under EUR600 and give me the total."
output4 = small_agent.run(goal4)

print("================ FINAL STRUCTURED OUTPUT (run 4, step-limit path) ================")
print(output4)


Starting Agent with Goal: 'Plan a 3-day trip to Porto under EUR600 and give me the total.'
Safety Check: Active. Max Step Limit: 1
--------------------------------------------------

[Step 1/1] Calling model...
Model chose tool: 'search_flights' with args: {'destination': 'Porto'}
Observation (Tool Output): {'destination': 'Porto (OPO)', 'price': 180.0}
Step limit reached before the agent finished.
================ FINAL STRUCTURED OUTPUT (run 4, step-limit path) ================
{"status": "failed", "reason": "step limit exceeded"}
